In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


Random 400 tấm ảnh 

In [ ]:
import os
import random
import shutil

# Đường dẫn
src_dir = "/kaggle/input/wildfire-prediction-dataset/valid/nowildfire"
dst_dir = "/kaggle/working/wildfire_images_500/train/nowildfire"

# Tạo folder đích nếu chưa có
os.makedirs(dst_dir, exist_ok=True)

# Lấy danh sách ảnh (lọc theo đuôi phổ biến)
images = [
    f for f in os.listdir(src_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

print("Tổng số ảnh:", len(images))

# Random 500 ảnh
selected_images = random.sample(images, 400)

# Copy ảnh
for img in selected_images:
    shutil.copy(
        os.path.join(src_dir, img),
        os.path.join(dst_dir, img)
    )

print("Đã copy xong ảnh")


Kiểm tra ảnh sau khi đã chia

In [ ]:
len(os.listdir(dst_dir))


In [ ]:
img_dir = "/kaggle/working/wildfire_images_500/train/nowildfire"

images = [
    f for f in os.listdir(img_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

print("Số ảnh:", len(images))


In [ ]:
import os
import matplotlib.pyplot as plt
from ipywidgets import Button, HBox, VBox, Output
from PIL import Image
index = 0
out = Output()

def show_image():
    out.clear_output()
    if index >= len(images):
        with out:
            print("Đã xem hết ảnh")
        return
    
    img_path = os.path.join(img_dir, images[index])
    img = Image.open(img_path)
    
    with out:
        plt.figure(figsize=(6,6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"{index+1}/{len(images)} - {images[index]}")
        plt.show()

def delete_image(b):
    global index
    img_path = os.path.join(img_dir, images[index])
    os.remove(img_path)
    print(f"Đã xoá: {images[index]}")
    images.pop(index)
    show_image()

def next_image(b):
    global index
    index += 1
    show_image()

btn_delete = Button(description="Xoá ảnh", button_style="danger")
btn_next = Button(description="Ảnh tiếp theo", button_style="success")

btn_delete.on_click(delete_image)
btn_next.on_click(next_image)

show_image()
VBox([out, HBox([btn_delete, btn_next])])


Tăng cường ảnh không cháy

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
INPUT_DIR  = "/kaggle/working/wildfire_images_500/train/nowildfire"
OUTPUT_DIR = "/kaggle/working/images_aug"
os.makedirs(OUTPUT_DIR, exist_ok=True)

images = [
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

print("Số ảnh gốc:", len(images))

flip + rotate

In [ ]:

def geometric_aug(img):
    h, w = img.shape[:2]

    # Flip
    if np.random.rand() < 0.5:
        img = cv2.flip(img, 1)
    if np.random.rand() < 0.5:
        img = cv2.flip(img, 0)

    # Rotate
    angle = np.random.uniform(-180, 180)
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    return img

for name in tqdm(images):
    img = cv2.imread(os.path.join(INPUT_DIR, name))
   
    aug = geometric_aug(img)
   





Brightness / Contrast / Saturation

In [ ]:
def color_aug(img):
    img = img.astype(np.float32)

    # Brightness
    beta = np.random.uniform(-30, 30)
    img += beta

    # Contrast
    alpha = np.random.uniform(0.8, 1.3)
    img *= alpha

    img = np.clip(img, 0, 255).astype(np.uint8)

    # Saturation (HSV)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hsv[...,1] = np.clip(
        hsv[...,1] * np.random.uniform(0.85, 1.2),
        0, 255
    )
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
for name in tqdm(images):
    img = cv2.imread(os.path.join(INPUT_DIR, name))
    aug = color_aug(img)
    cv2.imwrite(os.path.join(OUTPUT_DIR, f"color_{name}"), aug)




Noise & Blur

In [ ]:
def noise_blur_aug(img):
    # Gaussian noise
    noise = np.random.normal(0, 15, img.shape).astype(np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # Blur
    if np.random.rand() < 0.5:
        img = cv2.GaussianBlur(img, (5,5), 0)
    if np.random.rand() < 0.3:
        img = cv2.blur(img, (3,3))

    return img
for name in tqdm(images):
    img = cv2.imread(os.path.join(INPUT_DIR, name))
    aug = noise_blur_aug(img)
    cv2.imwrite(os.path.join(OUTPUT_DIR, f"noise_{name}"), aug)




Lọc sương

In [ ]:
def fog_aug(img):
    h, w = img.shape[:2]
    fog = np.zeros((h, w, 3), dtype=np.uint8)

    for _ in range(np.random.randint(1, 4)):
        x, y = np.random.randint(0, w), np.random.randint(0, h)
        r = np.random.randint(100, 300)
        cv2.circle(fog, (x, y), r, (255,255,255), -1)

    fog = cv2.GaussianBlur(fog, (151,151), 0)
    alpha = np.random.uniform(0.1, 0.3)

    return cv2.addWeighted(img, 1-alpha, fog, alpha, 0)
for name in tqdm(images):
    img = cv2.imread(os.path.join(INPUT_DIR, name))
    aug = fog_aug(img)
    cv2.imwrite(os.path.join(OUTPUT_DIR, f"fog_{name}"), aug)




In [ ]:
print("Tổng ảnh sau augmentation:", len(os.listdir("/kaggle/working/images_aug")))


Kiểm tra lại các ảnh no fire xem có giống fire không

In [ ]:
from tqdm import tqdm
FIRE_LIKE_DIR = "/kaggle/working/similar_fire"
SAFE_DIR = "/kaggle/working/safe_images_fire"

os.makedirs(FIRE_LIKE_DIR, exist_ok=True)
os.makedirs(SAFE_DIR, exist_ok=True)

In [ ]:
def fire_like_score(img):
    """
    Trả về tỉ lệ pixel có đặc trưng giống lửa
    """
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Ngưỡng màu lửa (đỏ/cam)
    lower_fire = np.array([0, 120, 120])
    upper_fire = np.array([25, 255, 255])

    mask = cv2.inRange(hsv, lower_fire, upper_fire)

    fire_pixels = cv2.countNonZero(mask)
    total_pixels = img.shape[0] * img.shape[1]

    return fire_pixels / total_pixels
THRESHOLD = 0.01  # 1%

for name in tqdm(os.listdir("/kaggle/working/images_aug")):
    if not name.lower().endswith(('.jpg', '.png', '.jpeg')):
        continue

    img_path = os.path.join("/kaggle/working/images_aug", name)
    img = cv2.imread(img_path)

    score = fire_like_score(img)

    if score > THRESHOLD:
        cv2.imwrite(os.path.join(FIRE_LIKE_DIR, name), img)
    else:
        cv2.imwrite(os.path.join(SAFE_DIR, name), img)




In [ ]:
print("Ảnh nghi giống lửa:", len(os.listdir(FIRE_LIKE_DIR)))
print("Ảnh non-fire an toàn:", len(os.listdir(SAFE_DIR)))


Kiểm tra lại các ảnh safe non fire

Chọn 1000 ảnh để import vào roboflow

In [ ]:
import os
import random
import shutil

# Đường dẫn
src_dir = "/kaggle/working/safe_images_fire"
dst_dir = "/kaggle/working/dataset_roboflow"

# Tạo folder đích nếu chưa có
os.makedirs(dst_dir, exist_ok=True)

# Lấy danh sách ảnh (lọc theo đuôi phổ biến)
images = [
    f for f in os.listdir(src_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

print("Tổng số ảnh:", len(images))

# Random 1000 ảnh
selected_images = random.sample(images, 1000)

# Copy ảnh
for img in selected_images:
    shutil.copy(
        os.path.join(src_dir, img),
        os.path.join(dst_dir, img)
    )


print("Đã copy xong 1000 ảnh")


In [ ]:
img_dir = "/kaggle/working/dataset_roboflow"

images = [
    f for f in os.listdir(img_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

print("Số ảnh:", len(images))

In [ ]:
import os
import matplotlib.pyplot as plt
from ipywidgets import Button, HBox, VBox, Output
from PIL import Image
index = 0
out = Output()

def show_image():
    out.clear_output()
    if index >= len(images):
        with out:
            print("🎉 Đã xem hết ảnh")
        return
    
    img_path = os.path.join(img_dir, images[index])
    img = Image.open(img_path)
    
    with out:
        plt.figure(figsize=(6,6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"{index+1}/{len(images)} - {images[index]}")
        plt.show()

def delete_image(b):
    global index
    img_path = os.path.join(img_dir, images[index])
    os.remove(img_path)
    print(f" Đã xoá: {images[index]}")
    images.pop(index)
    show_image()

def next_image(b):
    global index
    index += 1
    show_image()

btn_delete = Button(description=" Xoá ảnh", button_style="danger")
btn_next = Button(description=" Ảnh tiếp theo", button_style="success")

btn_delete.on_click(delete_image)
btn_next.on_click(next_image)

show_image()
VBox([out, HBox([btn_delete, btn_next])])


In [ ]:
!zip -r dataset_roboflow.zip /kaggle/working/dataset_roboflow

In [ ]:
import os
import shutil
import random

# ================= CẤU HÌNH =================
SOURCE_FOLDER = "/kaggle/working/dataset_roboflow"   # Tên folder chứa ảnh gốc của bạn
OUTPUT_FOLDER = "/kaggle/working/dataset_split"   # Tên folder kết quả đầu ra

def split_only_images():
    # 1. Kiểm tra folder gốc
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    # 2. Lấy danh sách toàn bộ ảnh
    all_files = [f for f in os.listdir(SOURCE_FOLDER) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    total_files = len(all_files)
    
    if total_files == 0:
        print(" Folder rỗng, không có ảnh nào!")
        return

    print(f"Tổng cộng: {total_files} ảnh. Đang chia...")
    
    # 3. Xáo trộn ngẫu nhiên
    random.shuffle(all_files)

    # 4. Tính toán số lượng cho từng tập
    train_count = int(total_files * 0.75)
    valid_count = int(total_files * 0.15)
    # Test lấy nốt phần còn lại để đảm bảo đủ 100%
    test_count = total_files - train_count - valid_count

    # 5. Chia danh sách
    datasets = {
        "train": all_files[:train_count],
        "valid": all_files[train_count : train_count + valid_count],
        "test":  all_files[train_count + valid_count :]
    }

    # 6. Thực hiện Copy
    if os.path.exists(OUTPUT_FOLDER):
        shutil.rmtree(OUTPUT_FOLDER) # Xóa folder cũ nếu có để làm mới

    for split_name, files in datasets.items():
        # Tạo folder đích: dataset_split/train, dataset_split/valid...
        target_dir = os.path.join(OUTPUT_FOLDER, split_name)
        os.makedirs(target_dir, exist_ok=True)
        
        print(f"   ↳ Đang copy {len(files)} ảnh vào '{split_name}'...")
        
        for file_name in files:
            src_path = os.path.join(SOURCE_FOLDER, file_name)
            dst_path = os.path.join(target_dir, file_name)
            shutil.copy2(src_path, dst_path)

    print("\n XONG! Cấu trúc folder mới:")
    print(f"    {OUTPUT_FOLDER}/")
    print(f"      ├──  train ({len(datasets['train'])} ảnh)")
    print(f"      ├──  valid ({len(datasets['valid'])} ảnh)")
    print(f"      └──  test  ({len(datasets['test'])} ảnh)")

if __name__ == "__main__":
    split_only_images()

In [ ]:
!pip install roboflow
import os
from roboflow import Roboflow

# ================= CẤU HÌNH =================
# Thư mục chứa 3 folder con: train, valid, test (do code trước tạo ra)
BASE_FOLDER = "dataset_split" 

# Thông tin Project của bạn (Lấy từ tin nhắn bạn gửi)
API_KEY = os.environ["ROBOFLOW_API_KEY"]
WORKSPACE = "fire-detection-lfqdr"
PROJECT = "wildfire_satelite-klbbn"

def upload_split_dataset():
    # 1. Kết nối Roboflow
    rf = Roboflow(api_key=API_KEY)
    try:
        project = rf.workspace(WORKSPACE).project(PROJECT)
        print(f" Đã kết nối tới Project: {PROJECT}")
    except Exception as e:
        print(f" Lỗi kết nối: {e}")
        return

    # 2. Định nghĩa các tập cần upload
    # Key là tên folder, Value là tên split trên Roboflow
    splits = {
        "train": "train",
        "valid": "valid",
        "test": "test"
    }

    # 3. Vòng lặp upload
    for folder_name, split_type in splits.items():
        folder_path = os.path.join(BASE_FOLDER, folder_name)
        
        if not os.path.exists(folder_path):
            print(f" Không tìm thấy folder: {folder_path}. Bỏ qua.")
            continue
            
        print(f"\n Đang upload tập '{split_type.upper()}' từ {folder_path}...")
        
        images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        for img in images:
            img_path = os.path.join(folder_path, img)
            
            try:
                # Upload lên project
                project.upload(
                    image_path=img_path,
                    annotation_path=None,    # None = Ảnh Null (Negative Sample)
                    split=split_type,        # Tự động chia vào train/valid/test
                    batch_name="negative_samples_upload" # Tên batch để dễ quản lý
                )
                print(f"   Shape: {split_type} | File: {img} -> OK")
            except Exception as e:
                print(f"    Lỗi {img}: {e}")

    print("\n HOÀN TẤT! Hãy vào Web Roboflow để kiểm tra.")
    print(" Nhớ Generate Version mới để gộp các ảnh này vào dataset nhé.")

if __name__ == "__main__":
    upload_split_dataset()